# **Procesamiento de Lenguaje Natural**
## Maestria en Inteligencia Artificial Aplicada
### Tecnologico de Monterrey

* **Nombres y matriculas**
    * Sarmiento Cervantes Jacqueline: A01795863
    * Mayoral Teran Alexandro: A01795899
* **Numero de equipo: 8**

---

# 🎓 Proyecto Integrador: Avance 4 - Evaluación Core, Pareto y Métricas Base (DISF)

Este documento es el **entregable** del Avance 4 y funge como la base teórica y técnica para la evaluación del sistema. Su estructura fluye de lo fundacional (arquitectura y telemetría) hacia la validación estadística (Pareto y Taxonomía de errores), cumpliendo con las exigencias de Banco de México y las rúbricas de la maestría.

---

## 1. Preparación del Entorno y Arquitectura Core

Para garantizar la reproducibilidad de este Notebook, inicializamos el entorno conectando directamente con la arquitectura modular desarrollada en `src/`.


In [ ]:
# 1. Preparación del Entorno
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv(project_root / ".env")
print("✅ Entorno preparado. Importaciones locales de src/ listas.")


### 1.1 Diseño y Arquitectura de Scripts Core (Deep-Dive)
Para automatizar la evaluación y cerrar las brechas críticas de avances pasados, se construyeron 3 pilares arquitectónicos:
1. **Multi-Chunk Handling y Síntesis Multi-Documento (`generador_ground_truth_llm.py`):** Banxico requiere cruzar información de múltiples regulaciones. Para lograr esto, se eliminó la restricción de un-solo-documento. Ahora el *Retriever* recupera el *Top-K* de fragmentos (Chunks) de distintas fuentes (CUB, LIC, etc.) y los inyecta simultáneamente en el prompt del LLM. Esto habilita el **Multi-Chunk Handling** real, permitiendo al modelo sintetizar una respuesta coherente a partir de múltiples piezas de rompecabezas normativo.
2. **Configuración Conmutable (`config_llm.py`):** Implementa el patrón *Factory*. Dependiendo de un flag en el `.env` (`USE_LOCAL_LLM=True`), conmuta instantáneamente entre Ollama (Llama 3.1) y OpenAI (GPT-4o) sin refactorizar el código base.
3. **El Orquestador (`evaluador_integral.py`):** Carga dinámicamente el `config_experimentos.json` con las arquitecturas a evaluar, inyecta el hash criptográfico, ejecuta el RAG y clasifica los fallos.

---

## 2. Rúbricas B3 y MA3: Trazabilidad, Versionamiento y Prompt Registry

Uno de los pilares de un proyecto MLOps para entidades financieras es la reproducibilidad. Si el sistema emite una respuesta, el Banco debe rastrear exactamente qué instrucción la generó. 
Se construyó el módulo `src/nlp_core/prompts_registry.py` respaldado por `prompts.json`. Al cargar un prompt, inyecta dinámicamente un **Hash Criptográfico (SHA-256)** que viaja con la telemetría.


In [ ]:
# 2. Demostración de Trazabilidad Criptográfica
from src.nlp_core.prompts_registry import get_prompt

# Obtenemos el prompt exacto de RAG y su Hash único
prompt_text, prompt_hash = get_prompt("rag_system")
print(f"🔒 Hash del Prompt Actual: {prompt_hash}")
print(f"📝 Inicio del Prompt: {prompt_text[:100]}...")


---

## 3. Módulo de Telemetría (TCO y P95)

En sistemas RAG, "el contexto es dinero". Se implementó la clase `RastreadorTelemetria` que intercepta las llamadas para contar los tokens a nivel BPE con `tiktoken` y calcular el TCO (Total Cost of Ownership).


In [ ]:
# 3. Inicialización del Rastreador de Telemetría
from src.nlp_core.telemetria import RastreadorTelemetria

telemetria = RastreadorTelemetria()
telemetria.iniciar_rastreo("Query_Ejemplo_01")
# (Aquí ocurre la ejecución del LLM)
telemetria.detener_rastreo()

print(f"⏱️ Latencia P95 estimada: {telemetria.obtener_latencia_promedio():.2f}s")


---

## 4. Evolución del Dataset, Contextual Retrieval y Diversidad de Modelos (MA1)

### 4.1 Modelos Alternativos y Residencia de Datos (Rúbrica MA1)
Para cumplir con **MA1**, configuramos ≥6 arquitecturas LLM distintas, asegurando que al menos un candidato fuera **Self-Hostable** (Llama 3.1 8B vía Ollama). Esto garantiza a Banxico la **residencia de datos** (on-premise) e independencia del *vendor lock-in*.

### 4.2 Ampliación del Golden Dataset (Rúbrica MA2 / B1)
El script de generación sintética llevó el *Golden Dataset* de 30 a **110 consultas evaluables**, garantizando significancia estadística.


In [ ]:
# 4. Carga del Golden Dataset Expandido
import json

ruta_dataset = project_root / "data" / "evaluacion_dataset.json"
with open(ruta_dataset, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(f"📚 Dataset cargado con {len(dataset)} consultas regulatorias complejas.")


### 4.3 Mitigación de Pérdida de Contexto (Context Loss)
Para evitar la "orfandad" semántica de fórmulas en incisos profundos, migramos a **Contextual Retrieval** (inyectando Títulos al texto antes de vectorizar) y **Búsqueda Híbrida RRF** (Coseno + BM25) para evitar la deriva semántica (*semantic drift*).

---

## 5. Prueba Ciega y Contaminación de Datos por Candidato (Rúbrica MA2)

Cumpliendo la rúbrica **MA2**, corrimos una prueba de *No-Context Test* **por cada candidato LLM**. Aislando al LLM del buscador, forzamos respuestas basadas solo en su corpus de entrenamiento original.
**Resultado:** Todos alucinaron regulaciones europeas, demostrando que el RAG es estrictamente necesario.


In [ ]:
# 5. Simulador de Prueba Ciega (Data Contamination)
from src.nlp_core.generacion import GeneradorRespuesta

generador_ciego = GeneradorRespuesta(llm_name="gpt-4o-mini", temperature=0.0)
pregunta_financiera = "¿Cómo se calcula la estimación preventiva de tarjetas de crédito según el Anexo 33?"

# LLM responde SIN fragmentos normativos recuperados
respuesta_ciega = generador_ciego.generar(pregunta_financiera, contexto_recuperado="")
print(f"⚠️ Respuesta Alucinada (Sin Contexto): {respuesta_ciega}")


---

## 6. Rúbricas MA4, MA5 y MA7: Optimización Multiobjetivo y Significancia Estadística

### 6.1 Significancia Estadística (Rúbrica MA5)
No comparamos números crudos. El orquestador ejecuta **1,000 resamples (Bootstrap CI)** sobre el NDCG para obtener intervalos de confianza al 95%. Si se solapan, es un empate estadístico.

### 6.2 Frontera de Pareto (Rúbrica MA4 y MA7)
Graficamos la Frontera mapeando **Costo Operativo/Latencia (Eje X)** vs **NDCG@10 (Eje Y)** para seleccionar el Top-2 óptimo.


In [ ]:
# 6. Renderizado de la Frontera de Pareto
from src.lab.graficos import plot_frontera_pareto
from pathlib import Path

# Buscamos el CSV de resultados más reciente generado por el Orquestador
carpeta_eval = project_root / "data" / "03_output" / "evaluaciones"
plot_frontera_pareto(carpeta_eval)
print("📈 Gráfico de Pareto generado exitosamente en data/03_output/pareto_frontier.png")


---

## 7. Rúbrica MA6: Taxonomía y Desagregación de Errores

Implementamos un flujo automático que desagrega cada fallo en 3 niveles (A/B/C):
- **A - Fallo de Recuperación:** El texto no llegó al Top K.
- **B - Fallo de Generación:** El LLM ignoró el texto o falló al razonar.
- **C - Fallo Estructural:** El LLM rompió el contrato JSON de Pydantic.


In [ ]:
# 7. Ejecución de Taxonomía de Errores
# (Representación del módulo de Streamlit y Pandas)
import pandas as pd

archivo_resultados = list(carpeta_eval.glob("ARENA_RESULTADOS*.csv"))[0]
df_resultados = pd.read_csv(archivo_resultados)

conteo_errores = df_resultados['error_type'].value_counts()
print("🚨 Taxonomía de Errores del Modelo Base:")
print(conteo_errores)


---
> **Conclusión del Avance 4:** El ecosistema de evaluación automatizada, la Frontera de Pareto con significancia estadística y la mitigación de *Context Loss* demuestran madurez técnica nivel *Enterprise*, justificando cuantitativamente las decisiones de arquitectura del motor RAG.
